In [14]:
from __future__ import annotations

import argparse
import collections
import json
import math
import os
import sys
import time
import uuid
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from urllib.parse import parse_qs, urlparse
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
# ---------------------------------------------------------------------------
# Paths (S3-first, mirrors labeling_chunks.py)
# ---------------------------------------------------------------------------

AWS_PROFILE = os.getenv("AWS_PROFILE", "PowerUserAccess-976193240866")
os.environ["AWS_PROFILE"] = AWS_PROFILE

S3_BUCKET = os.getenv("RAG_EXPERIMENT_S3_BUCKET", "codingrabbit-data-dev")

In [10]:
RAG_RESULTS_PATH = os.getenv(
    "RAG_EXPERIMENT_RESULTS_PATH",
    f"s3://{S3_BUCKET}/prepared/rag/experiments/outputs/experiment_results_cs50.json",
)

def _parse_s3_url(url: str) -> tuple[str, str | None]:
    s = str(url)
    if s.startswith("s3://"):
        rest = s[len("s3://"):]
        parts = rest.split("/", 1)
        return parts[0], parts[1] if len(parts) > 1 else None

    p = urlparse(s)
    if "console.aws.amazon.com" in p.netloc and "/s3/object/" in p.path:
        bucket = p.path.split("/s3/object/", 1)[1].strip("/ ")
        key = parse_qs(p.query).get("prefix", [None])[0]
        return bucket, key

    raise ValueError(f"Unrecognized S3 URL: {url}")


def _get_s3_client() -> Any:
    try:
        import boto3
        from botocore import UNSIGNED
        from botocore.client import Config
    except ModuleNotFoundError as e:
        raise RuntimeError("S3 paths require boto3/botocore. Install boto3 or use local paths.") from e

    if os.environ.get("S3_ANONYMOUS", "0") in ("1", "true", "True"):
        return boto3.client("s3", config=Config(signature_version=UNSIGNED))

    return boto3.Session(profile_name=AWS_PROFILE).client("s3")

In [11]:
def _read_text(path: str) -> str:
    """Read text from local or S3 path."""
    if path.startswith("s3://"):
        bucket, key = _parse_s3_url(path)
        if not key:
            raise ValueError(f"S3 object key not found in URL: {path}")
        s3 = _get_s3_client()
        try:
            obj = s3.get_object(Bucket=bucket, Key=key)
            return obj["Body"].read().decode("utf-8")
        except Exception as e:
            raise FileNotFoundError(f"Could not read s3://{bucket}/{key}: {e}") from e
    
    # Local file
    with open(path, encoding="utf-8") as f:
        return f.read()


# Load results
print(f"Loading results from: {RAG_RESULTS_PATH}")
results_text = _read_text(RAG_RESULTS_PATH)
results = json.loads(results_text)

print(f"\nLoaded {len(results)} experiment runs")
print(f"\nKeys in first result: {list(results[0].keys()) if results else 'N/A'}")

Loading results from: s3://codingrabbit-data-dev/prepared/rag/experiments/outputs/experiment_results_cs50.json

Loaded 80 experiment runs

Keys in first result: ['params', 'metrics', 'latency']


In [12]:
flattened = []
for result in results:
    row = {}
    # Add params with 'param_' prefix
    if 'params' in result:
        for key, value in result['params'].items():
            row[f'param_{key}'] = value
    # Add metrics with 'metric_' prefix
    if 'metrics' in result:
        for key, value in result['metrics'].items():
            row[f'metric_{key}'] = value
    flattened.append(row)

df = pd.DataFrame(flattened)

In [18]:
df.sort_values(by=['metric_f1','metric_recall'], ascending=False)

,param_embedding_model,param_top_k,param_rerank_strategy,param_rerank_lambda,param_fetch_multiplier,param_num_queries,param_num_chunks,param_collection_name,metric_recall,metric_precision,metric_f1,metric_recall@3,metric_precision@3,metric_f1@3,metric_mrr,metric_ndcg@3,metric_recall@5,metric_precision@5,metric_f1@5,metric_ndcg@5,metric_recall@8,metric_precision@8,metric_f1@8,metric_ndcg@8,metric_recall@10,metric_precision@10,metric_f1@10,metric_ndcg@10,metric_recall@15,metric_precision@15,metric_f1@15,metric_ndcg@15
28,sentence-transformers/multi-qa-mpnet-base-dot-v1,8,similarity,none,1,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.1405,0.1040,0.1153,NaN,NaN,NaN,0.2646,NaN,NaN,NaN,NaN,NaN,0.1405,0.1040,0.1153,0.1421,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32,sentence-transformers/multi-qa-mpnet-base-dot-v1,10,similarity,none,1,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.1549,0.0920,0.1116,NaN,NaN,NaN,0.2685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1549,0.092,0.1116,0.1467,NaN,NaN,NaN,NaN
36,sentence-transformers/multi-qa-mpnet-base-dot-v1,15,similarity,none,1,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.1981,0.0797,0.1104,NaN,NaN,NaN,0.2722,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1981,0.0797,0.1104,0.166
31,sentence-transformers/multi-qa-mpnet-base-dot-v1,8,mmr_0.9,0.9,4,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.1319,0.0973,0.1079,NaN,NaN,NaN,0.2601,NaN,NaN,NaN,NaN,NaN,0.1319,0.0973,0.1079,0.1370,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,sentence-transformers/multi-qa-mpnet-base-dot-v1,5,similarity,none,1,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.1003,0.1239,0.1067,NaN,NaN,NaN,0.2534,NaN,0.1003,0.1239,0.1067,0.1374,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49,BAAI/bge-base-en-v1.5,8,mmr_0.5,0.5,4,113,3078,exp_BAAI_bge_base_en_v1.5,0.0354,0.0277,0.0296,NaN,NaN,NaN,0.0801,NaN,NaN,NaN,NaN,NaN,0.0354,0.0277,0.0296,0.0361,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,BAAI/bge-base-en-v1.5,5,mmr_0.7,0.7,4,113,3078,exp_BAAI_bge_base_en_v1.5,0.0269,0.0354,0.0292,NaN,NaN,NaN,0.0816,NaN,0.0269,0.0354,0.0292,0.0387,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,BAAI/bge-base-en-v1.5,5,mmr_0.5,0.5,4,113,3078,exp_BAAI_bge_base_en_v1.5,0.0208,0.0283,0.0232,NaN,NaN,NaN,0.0739,NaN,0.0208,0.0283,0.0232,0.0327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,BAAI/bge-base-en-v1.5,3,mmr_0.7,0.7,4,113,3078,exp_BAAI_bge_base_en_v1.5,0.0151,0.0354,0.0204,0.0151,0.0354,0.0204,0.0737,0.0385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# df.sort_values(by='metric_f1', ascending=False)

,param_embedding_model,param_top_k,param_rerank_strategy,param_rerank_lambda,param_fetch_multiplier,param_num_queries,param_num_chunks,param_collection_name,metric_recall,metric_precision,...,metric_f1@8,metric_ndcg@8,metric_recall@10,metric_precision@10,metric_f1@10,metric_ndcg@10,metric_recall@15,metric_precision@15,metric_f1@15,metric_ndcg@15
28,sentence-transformers/multi-qa-mpnet-base-dot-v1,8,similarity,none,1,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.0815,0.0675,...,0.0714,0.0854,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,sentence-transformers/multi-qa-mpnet-base-dot-v1,15,similarity,none,1,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.1167,0.0502,...,NaN,NaN,NaN,NaN,NaN,NaN,0.1167,0.0502,0.0684,0.0984
31,sentence-transformers/multi-qa-mpnet-base-dot-v1,8,mmr_0.9,0.9,4,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.0754,0.0631,...,0.0665,0.0799,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32,sentence-transformers/multi-qa-mpnet-base-dot-v1,10,similarity,none,1,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.0830,0.0549,...,NaN,NaN,0.0830,0.0549,0.0640,0.0838,NaN,NaN,NaN,NaN
35,sentence-transformers/multi-qa-mpnet-base-dot-v1,10,mmr_0.9,0.9,4,113,3078,exp_sentence_transformers_multi_qa_mpnet_base_...,0.0815,0.0540,...,NaN,NaN,0.0815,0.0540,0.0629,0.0809,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43,BAAI/bge-base-en-v1.5,3,mmr_0.9,0.9,4,113,3078,exp_BAAI_bge_base_en_v1.5,0.0183,0.0383,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,all-MiniLM-L6-v2,3,mmr_0.5,0.5,4,113,3078,exp_all_MiniLM_L6_v2,0.0179,0.0383,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,BAAI/bge-base-en-v1.5,8,mmr_0.5,0.5,4,113,3078,exp_BAAI_bge_base_en_v1.5,0.0233,0.0210,...,0.0214,0.0272,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,BAAI/bge-base-en-v1.5,3,mmr_0.5,0.5,4,113,3078,exp_BAAI_bge_base_en_v1.5,0.0156,0.0354,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
